## 构建你自己的代码解释器——使用 o3-mini 动态生成并执行工具

赋予 LLM 智能体与外部世界或其他智能体交互能力的核心是“工具（或函数）调用”，即 LLM 可以带参数调用一个函数（一段代码）。通常，这些函数由开发者预先定义，并明确其预期的输入和输出。然而，在本 Cookbook 中，我们探索一种更灵活的范式——使用 LLM 模型（这里是 **o3-mini**）**动态生成工具**，并能够使用代码解释器执行该工具。

### 借助代码解释器进行动态生成的工具调用
动态生成的工具是 LLM 在运行时根据用户提示自行创建的函数或代码块。这意味着你不必在代码库中预先定义每一种可能的场景——从而能够以更开放、更有创造性、更具适应性的方式解决问题。

动态生成的工具调用更进一步，赋予 LLM 即时生成工具并执行代码块的能力。这种动态方法对以下任务尤其有用：

- 数据分析与可视化
- 数据处理与转换
- 机器学习工作流的生成与执行
- 流程自动化与脚本编写
- 以及更多随着实验不断涌现的新可能

### 使用 o3-mini 动态生成工具

o3-mini 模型于 2025 年 1 月 31 日发布，具有出色的 STEM 能力——尤其擅长科学、数学和编程——同时保持了小型模型的低成本和低延迟。在本 Cookbook 中，我们将演示 o3-mini 生成 Python 代码来解读数据并得出洞见的能力。

推理模型尤其擅长生成用于分析数据的动态工具，因为它们可以自行推理，无需显式的思维链提示。事实上，提供显式的思维链指令反而可能干扰模型的内部推理，导致次优结果。你可以在[这里](https://openai.com/index/openai-o3-mini/)了解更多关于 o3-mini 的信息。

### 为什么要构建自己的代码解释器

许多 API 提供方——例如 OpenAI 的 Assistants API——都提供内置的代码解释器功能。这些内置代码解释器可能非常强大，但在某些情况下，开发者可能需要创建自己的自定义代码解释器。例如：

1. **语言或库支持**：内置解释器可能不支持你的任务所需的特定编程语言（例如 C++、Java 等）或库。
2. **任务兼容性**：你的用例可能与提供方的内置方案不兼容。
3. **模型限制**：你可能需要使用提供方解释器不支持的语言模型。
4. **成本考量**：代码执行或模型使用的成本结构可能不符合你的预算或限制。
5. **文件大小**：输入数据的文件过大，或不受提供方解释器支持。
6. **与内部系统集成**：提供方的解释器可能无法与你的内部系统集成。

### 你将学到什么
通过学习本 Cookbook，你将学会：

- 使用 Docker 搭建一个隔离的 Python 代码执行环境
- 为 LLM 智能体配置你自己的代码解释器工具
- 为保障安全，清晰地分离各“智能体”的职责
- 使用 **o3-mini** 模型动态生成用于数据分析的代码
- 编排智能体以高效完成给定任务
- 设计一个能够动态生成并执行代码的智能体应用

你将学习如何从零开始构建自定义代码解释器工具，利用 LLM 的能力生成复杂代码，并在隔离环境中安全地执行这些代码——所有这些都是为了让你的 AI 驱动应用更灵活、更强大、更具成本效益。

### 示例场景

我们将使用 [Key Factors Traffic Accidents](https://www.kaggle.com/datasets/willianoliveiragibin/key-factors-traffic-accidents) 提供的示例数据来回答一组问题。这些问题无需预先定义，我们会赋予 LLM 生成代码来回答此类问题的能力。 

示例问题可以是： 
- 哪些因素对事故频率的影响最大？（特征重要性分析）
- 哪些区域的事故风险最高？（分类/聚类）
- 交通罚款金额如何影响事故数量？（回归/因果推断）
- 能否确定降低事故率的最佳罚款金额？（优化模型）
- 更高的罚款是否与更低的平均车速或更少的事故相关？（相关性/回归）
- 等等……

如果使用传统的**预定义工具调用**方法，开发者需要为每个问题预先定义函数。这限制了 LLM 回答预定义函数集合之外其他问题的能力。我们采用**动态工具调用**方法来克服这一限制：由 LLM 生成代码，并使用代码解释器工具执行这些代码。 

## 概览
我们来深入了解构建这个基于动态生成工具调用的智能体应用的步骤。该应用由三个部分组成：

#### 第 1 步：搭建隔离的代码执行容器环境

我们需要一个安全的环境来执行 LLM 生成的函数调用。我们希望避免直接在主机上运行 LLM 生成的代码，因此将创建一个资源访问受限（例如无网络访问）的 Docker 容器环境。默认情况下，Docker 容器无法访问主机的文件系统，这有助于确保 LLM 生成的任何代码都被限制在容器内。 

##### ⚠️ 警告：为 LLM 生成的代码实施严格的防护措施
LLM 可能会生成带来意外后果的有害代码。作为最佳实践，应隔离代码执行环境，只授予任务所需的资源访问权限。避免在你的主机或笔记本电脑上运行 LLM 生成的代码。 

#### 第 2 步：定义并测试智能体

“**什么是智能体？**”在本 Cookbook 的语境中，智能体是指：
1. 供 LLM 遵循的一组指令，即开发者提示
2. 一个 LLM 模型，以及通过 API 调用该模型的能力
3. 对某个函数的工具调用权限，以及执行该函数的能力 

我们将定义两个智能体：
1. FileAccessAgent：该智能体读取文件，并将上下文提供给 PythonCodeExecAgent。
2. PythonCodeExecAgent：该智能体生成用于回答用户问题的 Python 代码，并在 Docker 容器中执行这些代码。

#### 第 3 步：设置智能体编排以运行应用 
根据应用需求，编排智能体的方式有很多种。在本例中，我们将使用一种简单的编排方式：由用户提供任务，然后依次调用各个智能体来完成任务。  

整体编排如下图所示：

![](../../images/oo_aa_image_1_code_interpreter_agents.png)

## 开始动手


### 前提条件
开始之前，请确保在你的主机上安装并配置好以下内容：

1. Docker：已在本地机器上安装并运行。你可以了解更多关于 Docker 的信息，并[从这里安装](https://www.docker.com/)。 
2. Python：已在本地机器上安装。你可以了解更多关于 Python 的信息，并[从这里安装](https://www.python.org/downloads/)。 
3. OpenAI API 密钥：已在本地机器上设置为环境变量，或设置在根目录的 .env 文件中。你可以了解更多关于 OpenAI API 密钥的信息，并[从这里进行设置](https://platform.openai.com/docs/api-reference/introduction)。 


### 第 1 步：搭建隔离的代码执行环境 

我们来定义一个用于执行代码的 Docker 容器环境。我已经在 `resources/docker` 目录下定义了 **[dockerfile](https://github.com/openai/openai-cookbook/blob/main/examples/object_oriented_agentic_approach/resources/docker/dockerfile)**，它将用于创建具有以下规格的容器环境：
- 以 Python 3.10 为基础 
- 一个非 root 用户 
- 预装 requirements.txt 中的软件包  

创建 docker 镜像过程中包含的 requirements.txt 列出了 LLM 生成的代码完成任务时可能需要的所有软件包。由于我们会禁止容器访问网络，因此需要预先安装任务所需的软件包。出于安全考虑，LLM 不被允许安装任何额外的软件包。 

你可以根据语言要求（例如 Python 3.10）创建自己的 docker 镜像，并预装任务所需的软件包；也可以针对特定语言（例如 Java、C++ 等）和任务所需的软件包创建自定义 docker 镜像。 

我们使用以下命令构建 docker 镜像。为简洁起见，我将输出重定向并用 grep 过滤成功消息，如果构建失败则打印一条消息。

In [1]:
!docker build -t python_sandbox:latest ./resources/docker 2>&1 | grep -E "View build details|ERROR" || echo "Build failed."

View build details: docker-desktop://dashboard/build/desktop-linux/desktop-linux/kl8fo02q7rgbindi9b42pn1zr


我们以受限模式运行容器。容器将在后台运行。这正是为容器定义安全策略的时机。良好的做法是只为容器开放任务所需的最少功能。默认情况下，容器内部无法访问主机文件系统。我们还要限制它的网络访问，使其无法访问互联网或任何其他网络资源。 

In [2]:
# Run the container in restricted mode. The container will run in the background.
!docker run -d --name sandbox --network none --cap-drop all --pids-limit 64 --tmpfs /tmp:rw,size=64M   python_sandbox:latest sleep infinity

8446d1e9a7972f2e00a5d1799451c1979d34a2962aa6b4c35a9868af8d321b0e


我们使用 `docker ps` 确认容器正在运行，该命令应会列出我们的容器。 

In [3]:
!docker ps 

CONTAINER ID   IMAGE                   COMMAND            CREATED         STATUS         PORTS     NAMES
8446d1e9a797   python_sandbox:latest   "sleep infinity"   2 seconds ago   Up 2 seconds             sandbox


### 第 2 步：定义并测试智能体

出于我们的目的，我们将定义两个智能体。 
1.	**智能体 1：文件访问智能体（使用预定义工具调用）**
- 包含理解文件内容的指令，以便将其作为上下文提供给智能体 2。
- 可以访问主机的文件系统。 
- 可以从主机读取文件并将其复制到 Docker 容器中。
- 无法访问代码解释器工具。
- 使用 gpt-4o 模型。

2.	**智能体 2：Python 代码生成与执行器（使用动态生成的工具调用和代码执行）**
- 从智能体 1 接收文件内容的上下文。
- 包含生成 Python 脚本以回答用户问题的指令。
- 可以访问 Docker 容器内的代码解释器，用于执行 Python 代码。
- 只能访问 Docker 容器内的文件系统（而非主机）。
- 无法访问主机的文件系统或网络。
- 使用我们最新的、擅长代码生成的 **o3-mini** 模型。

将文件访问（智能体 1）与代码生成和执行（智能体 2）的职责分离，对于防止 LLM 直接访问或修改主机至关重要。 


**由于智能体 1 可以访问主机文件系统，应将其限制为静态工具调用。**  


| 智能体 | 工具调用类型 | 访问主机文件系统 | 访问 Docker 容器文件系统 | 访问代码解释器 |
|-------|-------------------|----------------------------|----------------------------------------|----------------------------|
| 智能体 1：文件访问 | 预定义工具 | 是 | 是 | 否 |
| 智能体 2：Python 代码生成与执行器 | 动态生成的工具 | 否 | 是 | 是 |


为了让智能体和工具保持条理清晰，我们按照面向对象编程原则定义了一组**核心类**，用于一致地创建这两个智能体。

- **BaseAgent**：我们从一个抽象基类开始，它强制规定 `task()` 等通用方法签名。基类还提供用于调试的 logger、语言模型接口，以及 `add_context()`（用于为智能体添加上下文）等其他通用函数。 
- **ChatMessages**：由于 ChatCompletions API 是无状态的，该类用于存储对话历史。 
- **ToolManager**：用于管理智能体可调用工具的类。
- **ToolInterface**：智能体可调用的任何“工具”的抽象类，使所有工具具有一致的接口。

这些类定义在 [object_oriented_agents/core_classes](./resources/object_oriented_agents/core_classes) 目录中。 

#### 核心类的 UML 类图
下面的类图展示了各核心类之间的关系。该 UML（统一建模语言）图是使用 [Mermaid](https://mermaid) 生成的

![](../../images/oo_aa_image_2_uml_diagram.png)

**定义智能体 1：带有 FileAccessTool 的 FileAccessAgent**

我们先定义继承自 ToolInterface 类的 FileAccessTool。**FileAccessTool** 工具定义在 `resources/registry/tools` 目录下的 [file_access_tool.py](https://github.com/openai/openai-cookbook/blob/main/examples/object_oriented_agentic_approach/resources/registry/tools/file_access_tool.py) 文件中。

- FileAccessTool 实现了 ToolInterface 类，确保工具具有一致的接口。 
- 将用于 OpenAI Function Calling API 的工具定义绑定在 `get_definition` 方法中，并与工具的 `run` 方法放在一起，可以确保可维护性、可扩展性和可复用性。 

现在，我们定义扩展了 BaseAgent 类的 **FileAccessAgent**，并将 **FileAccessTool** 绑定到该智能体。FileAccessAgent 定义在 `resources/registry/agents` 目录下的 [file_access_agent.py](https://github.com/openai/openai-cookbook/blob/main/examples/object_oriented_agentic_approach/resources/registry/agents/file_access_agent.py) 文件中。FileAccessAgent：

- 是 BaseAgent 类的一个具体实现。 
- 使用开发者提示、模型名称、logger 和语言模型接口进行初始化。如有需要，开发者可以覆盖这些值。 
- 具有一个 setup_tools 方法，用于将 FileAccessTool 注册到工具管理器。 
- 具有一个 `task` 方法，它调用 FileAccessTool 读取文件，并将上下文提供给 PythonCodeExecAgent。
- 使用 `model_name='gpt-4o'`，它为该任务提供了足够的推理和工具调用能力。


**定义智能体 2：带有 PythonExecTool 的 PythonExecAgent**  

类似地，PythonExecTool 继承自 ToolInterface 类，并实现了 get_definition 和 run 方法。get_definition 方法以 OpenAI Function Calling API 所期望的格式返回工具定义。run 方法在 Docker 容器中执行 Python 代码并返回输出。该工具定义在 `resources/registry/tools` 目录下的 [python_code_interpreter_tool.py](https://github.com/openai/openai-cookbook/blob/main/examples/object_oriented_agentic_approach/resources/registry/tools/python_code_interpreter_tool.py) 文件中。

同样，PythonExecAgent 是 BaseAgent 类的一个具体实现。它定义在 `resources/registry/agents` 目录下的 [python_code_exec_agent.py](https://github.com/openai/openai-cookbook/blob/main/examples/object_oriented_agentic_approach/resources/registry/agents/python_code_exec_agent.py) 文件中。PythonExecAgent：

- 是 BaseAgent 类的一个具体实现。 
- 使用开发者提示、模型名称、logger 和语言模型接口进行初始化。如有需要，开发者可以覆盖这些值。 
- 具有一个 setup_tools 方法，用于将 PythonExecTool 注册到工具管理器。 
- 具有一个 `task` 方法，它调用 OpenAI API 来执行用户的任务；在本例中，即生成一个 Python 脚本来回答用户的问题，并使用代码解释器工具运行它。
- 使用 `model_name='o3-mini'`，该模型擅长代码生成等 STEM 任务。
- 使用 `reasoning_effort='high'`，鉴于任务的复杂性，它能进行更完整的推理，代价是生成更多 token、响应更慢。默认值为 medium，在速度和推理准确性之间取得平衡。

你可以在[这里](https://platform.openai.com/docs/guides/reasoning)了解更多关于 `reasoning_effort` 参数的信息。

### 第 3 步：设置智能体编排以运行应用 

定义好智能体后，现在可以定义运行应用的编排循环。该循环会提示用户输入问题或任务，然后调用 FileAccessAgent 读取文件，并将上下文提供给 PythonExecAgent。PythonExecAgent 将生成用于回答用户问题的 Python 代码，并在 Docker 容器中执行这些代码。代码执行的输出会展示给用户。 

用户可以输入 'exit' 来停止应用。我们的问题是：**What factors contribute the most to accident frequency?** 请注意，我们并没有预先定义回答这个问题的函数。



In [4]:
# Import the agents from registry/agents

from resources.registry.agents.file_access_agent import FileAccessAgent
from resources.registry.agents.python_code_exec_agent import PythonExecAgent


prompt = """Use the file traffic_accidents.csv for your analysis. The column names are:
Variable	Description
accidents	Number of recorded accidents, as a positive integer.
traffic_fine_amount	Traffic fine amount, expressed in thousands of USD.
traffic_density	Traffic density index, scale from 0 (low) to 10 (high).
traffic_lights	Proportion of traffic lights in the area (0 to 1).
pavement_quality	Pavement quality, scale from 0 (very poor) to 5 (excellent).
urban_area	Urban area (1) or rural area (0), as an integer.
average_speed	Average speed of vehicles in km/h.
rain_intensity	Rain intensity, scale from 0 (no rain) to 3 (heavy rain).
vehicle_count	Estimated number of vehicles, in thousands, as an integer.
time_of_day	Time of day in 24-hour format (0 to 24).
accidents	traffic_fine_amount
"""


print("Setup: ")
print(prompt)

print("Setting up the agents... ")

# Instantiate the agents with the default constructor defined values
# Developer may override the default values - prompt, model, logger, and language model interface if needed

# This agent use gpt-4o by default
file_ingestion_agent = FileAccessAgent()

# Let's make sure agent uses o3-mini model and set the reasoning_effort to high
data_analysis_agent = PythonExecAgent(model_name='o3-mini', reasoning_effort='high')

print("Understanding the contents of the file...")
# Give a task to the file ingestion agent to read the file and provide the context to the data analysis agent 
file_ingestion_agent_output = file_ingestion_agent.task(prompt)

# Add the file content as context to the data analysis agent
# The context is added to the agent's tool manager so that the tool manager can use the context to generate the code 

data_analysis_agent.add_context(prompt)
data_analysis_agent.add_context(file_ingestion_agent_output)

while True:

    print("Type your question related to the data in the file. Type 'exit' to exit.")
    user_input = input("Type your question.")

    if user_input == "exit":
        print("Exiting the application.")
        break

    print(f"User question: {user_input}")

    print("Generating dynamic tools and using code interpreter...")
    data_analysis_agent_output = data_analysis_agent.task(user_input)

    print("Output...")
    print(data_analysis_agent_output)


Setup: 
Use the file traffic_accidents.csv for your analysis. The column names are:
Variable	Description
accidents	Number of recorded accidents, as a positive integer.
traffic_fine_amount	Traffic fine amount, expressed in thousands of USD.
traffic_density	Traffic density index, scale from 0 (low) to 10 (high).
traffic_lights	Proportion of traffic lights in the area (0 to 1).
pavement_quality	Pavement quality, scale from 0 (very poor) to 5 (excellent).
urban_area	Urban area (1) or rural area (0), as an integer.
average_speed	Average speed of vehicles in km/h.
rain_intensity	Rain intensity, scale from 0 (no rain) to 3 (heavy rain).
vehicle_count	Estimated number of vehicles, in thousands, as an integer.
time_of_day	Time of day in 24-hour format (0 to 24).
accidents	traffic_fine_amount

Setting up the agents... 
Understanding the contents of the file...


2025-02-03 13:03:54,066 - MyApp - INFO - Handling tool call: safe_file_access
2025-02-03 13:03:54,067 - MyApp - INFO - Tool arguments: {'filename': './resources/data/traffic_accidents.csv'}
2025-02-03 13:03:54,562 - MyApp - INFO - Tool 'safe_file_access' response: Copied ./resources/data/traffic_accidents.csv into sandbox:/home/sandboxuser/.
The file content for the first 15 rows is:
    accidents  traffic_fine_amount  traffic_density  traffic_lights  pavement_quality  urban_area  average_speed  rain_intensity  vehicle_count  time_of_day
0          20               4.3709           2.3049         753.000            0.7700           1        321.592          1.1944       290.8570     160.4320
1          11               9.5564           3.2757           5.452            4.0540           1        478.623          6.2960       931.8120       8.9108
2          19               7.5879           2.0989           6.697          345.0000           0        364.476          2.8584       830.086

Type your question related to the data in the file. Type 'exit' to exit.
User question: What factors contribute the most to accident frequency?
Generating dynamic tools and using code interpreter...


2025-02-03 13:04:39,427 - MyApp - INFO - Handling tool call: execute_python_code
2025-02-03 13:04:39,429 - MyApp - INFO - Tool arguments: {'python_code': "import pandas as pd\nimport numpy as np\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.preprocessing import StandardScaler\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n# Load the dataset\nfile_path = '/home/sandboxuser/traffic_accidents.csv'\ndf = pd.read_csv(file_path)\n\n# Show basic information\nprint('Dataset shape:', df.shape)\nprint('First few rows:')\nprint(df.head(), '\\n')\nprint('Columns:', df.columns.tolist(), '\\n')\n\n# Correlation matrix analysis\ncorr_matrix = df.corr()\nprint('Correlation matrix:')\nprint(corr_matrix, '\\n')\n\n# Correlation of each feature with accidents\nacc_corr = corr_matrix['accidents'].drop('accidents').sort_values(key=lambda x: abs(x), ascending=False)\nprint('Correlation of other variables with accidents (sorted by absolute correlation):')\nprint(acc_corr, '\\n'

Output...
The analysis shows that one variable stands out by far:

• Both the simple correlation analysis and regression results indicate that traffic_fine_amount is the dominant factor—its correlation with accidents is strong (about –0.75), and in the standardized linear regression its coefficient is the largest in magnitude (around –3.89). The negative sign suggests that, in this data, higher fine amounts are associated with fewer accidents (which might reflect more stringent enforcement or deterrence).

Other findings include:

• The Random Forest model also ranks traffic_fine_amount as most important (importance ≈ 0.58), with the next most influential factor being traffic_density (importance ≈ 0.17). Although its simple correlation with accidents is lower, traffic_density may contribute non‐linearly.

• Additional factors like urban_area, average_speed, rain_intensity, and time_of_day have moderate associations (with linear model coefficients ranging between about ±0.5 to +0.74). T

在这个示例中，**o3-mini** 根据用户的问题动态生成了一个用于分析数据的工具（Python 脚本）。请注意，**o3-mini** 使用了多种方法来研究这个问题，例如相关性分析、线性回归和随机森林模型。这种方法突出了以下几点：

**reasoning_effort**：模型所进行推理的深度（例如本例中所采用方法的数量），通常会随着该参数从 low、medium 提高到 high 而增加。你可以尝试不同的推理强度级别，看看有何差异。

**动态生成的工具调用**：用于分析数据的工具（Python 脚本）并非由开发者手动编写或预先确定，而是由 o3-mini 模型在运行时创建了相关的数据探索和相关性分析代码。

**隔离的代码执行**：为确保安全并避免在主机上运行不受信任的代码，该 Python 脚本是使用 `execute_python_code` 工具在 Docker 容器内执行的。该容器的资源访问受到限制（例如无网络访问、文件系统访问有限），从而将任意代码执行可能带来的风险降到最低。


### 结论

本 Cookbook 提供了一份指南，介绍如何开发针对特定应用需求量身定制的**自定义代码解释器**，以解决厂商提供的方案中存在的局限，例如语言限制、成本考量，以及需要灵活搭配不同 LLM 或模型等。

**管理智能体和工具的方法**：我们还定义了一组核心类来管理智能体和工具。这种方法确保智能体和工具具有一致的接口，并可在不同应用之间复用。可以创建一个智能体和工具的仓库（例如 [registry](https://github.com/openai/openai-cookbook/tree/main/examples/object_oriented_agentic_approach/resources/registry) 文件夹）来管理它们。

**o3-mini 模型**：我们展示了 o3-mini 模型能够根据用户极简的提示，在运行时生成复杂代码来分析数据。随后，o3-mini 模型对分析结果进行推理，向用户解释这些结果。

最后，**回顾一下**，构建具有动态工具调用的智能体应用的三个步骤是：
1. 搭建隔离的代码执行容器环境
2. 定义并测试智能体
3. 设置智能体编排以运行应用 

我们讨论了隔离代码执行环境的重要性，以确保安全并避免在主机上运行不受信任的代码。我们以一个 CSV 文件为用例，演示了如何动态生成工具（Python 脚本）来分析数据并回答用户的问题。我们还展示了如何在 Docker 容器中执行代码并将输出返回给用户。